In [ ]:
import numpy as np

class NaiveBayesClassifier:
    def __init__(self, alpha=1.0):
        self.alpha = alpha # Laplace smoothing factor
        self.vocab = {}
        self.log_prior_spam = 0
        self.log_prior_ham = 0
        self.log_likelihood_spam = None
        self.log_likelihood_ham = None

    def fit(self, X_text, y):
    # 1. Building vocab from X_text strings
        for msg in (X_text):
            msg=msg.lower().strip()
            msg=msg.split()
            for  word in msg:
                    if word not in self.vocab:
                      self.vocab[word]=len(self.vocab)
        # 2. Convert X_text to count matrices
        num_msg=len(X_text)
        vocab_size=len(self.vocab)
        X_matrix=np.zeros((num_msg,vocab_size))
        for msg_idx,msg in enumerate(X_text):
            msg=msg.lower().strip().split()
            for word in msg:
                if word in self.vocab:
                    word_idx=self.vocab[word]
                    X_matrix[msg_idx,word_idx]+=1
        # 3. Compute log priors
        length=len(y)
        ones=np.sum(y==1)
        zeroes=np.sum(y==0)
        P_spam=ones/num_msg
        P_ham=zeroes/num_msg
        P_spam=np.log(P_spam)
        P_ham=np.log(P_ham)
        self.log_prior_spam=P_spam
        self.log_prior_ham=P_ham
        # 4. Compute smoothed log likelihood arrays
        # Extract all rows from X_matrix where the message is classified as Spam (y == 1)
        spam_messages_matrix = X_matrix[y == 1]

        # Extract all rows from X_matrix where the message is classified as Ham (y == 0)
        ham_messages_matrix = X_matrix[y == 0]
        spam_wrd_count=np.sum(spam_messages_matrix, axis=0)
        ham_wrd_count=np.sum(ham_messages_matrix, axis=0)
        total_of_all_words_in_spam=np.sum(spam_wrd_count)
        total_of_all_words_in_ham=np.sum(ham_wrd_count)
        denom_spam=(total_of_all_words_in_spam)+(self.alpha*vocab_size)
        denom_ham=(total_of_all_words_in_ham)+(self.alpha*vocab_size)
        Prob_of_word_spam=(spam_wrd_count+self.alpha)/denom_spam
        Prob_of_word_ham=(ham_wrd_count+self.alpha)/denom_ham
        self.log_likelihood_spam=np.log(Prob_of_word_spam)
        self.log_likelihood_ham=np.log(Prob_of_word_ham)

    def predict(self, X_text):
        predictions=[]
        for msg in X_text:
            msg=msg.lower().strip().split()
             
        # Step 1: Initialize your scores to the priors for THIS specific message
            spam_score = self.log_prior_spam
            ham_score = self.log_prior_ham
            
            for word in msg:
                if word in self.vocab:
                    word_idx = self.vocab[word]
                    
                    # Step 2: Add the individual word log-likelihoods to your scores
                    spam_score += self.log_likelihood_spam[word_idx]
                    ham_score += self.log_likelihood_ham[word_idx]
            
            # Step 3: Make the final choice based on which score is higher
            if spam_score > ham_score:
                predictions.append(1)  # Classify as Spam
            else:
                predictions.append(0)  # Classify as Ham (Legitimate)
                
        # Return the final results array
        return np.array(predictions)
    # 1. Create a tiny mock training dataset (2 Spam messages, 2 Ham messages)
    train_data = [
        "WINNER claim your free lottery cash prize now",
        "Urgent text code to claim free money prize",
        "Hey dad can you pick me up after school tomorrow",
        "Are we still meeting up for dinner tonight"
    ]
    # 1 = Spam, 0 = Ham
    train_labels = np.array([1, 1, 0, 0])

    # 2. Initialize and Train the model
    clf = NaiveBayesClassifier(alpha=1.0)
    clf.fit(train_data, train_labels)
    print("--- Training Successful! ---")
    print(f"Vocabulary Size Learned: {len(clf.vocab)} unique words.")

    # 3. Create brand new test sentences the model has NEVER seen
    test_data = [
        "Free cash prize winner click here",  # Expect: 1 (Spam)
        "Hey text me back tomorrow morning"   # Expect: 0 (Ham)
    ]

    # 4. Predict
    results = clf.predict(test_data)
    print("\nTest Results:")
    for message, pred in zip(test_data, results):
        label = "SPAM" if pred == 1 else "HAM (Legitimate)"
        print(f"Message: '{message}' --> Prediction: {label}")


--- Training Successful! ---
Vocabulary Size Learned: 30 unique words.

Test Results:
Message: 'Free cash prize winner click here' --> Prediction: SPAM
Message: 'Hey text me back tomorrow morning' --> Prediction: HAM (Legitimate)
